# Team 08 - Mukhesh: real-image detector cache + Agent C (A2C)
**Part 1 needs Runtime -> Change runtime type -> T4 GPU.** Part 2 (Agent C) runs on CPU (fine in the same GPU session).

Before starting: `team08` folder must be in YOUR My Drive (Shared with me -> right-click team08 -> Organize -> Add shortcut -> My Drive).

## Step 1 - Mount, unpack, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/team08'
!mkdir -p $DRIVE/results
!rm -rf /content/uav_inspection
!unzip -o -q $DRIVE/uav_inspection.zip -d /content/
%cd /content/uav_inspection
!pip install -q gymnasium stable-baselines3 tensorboard
!ls scripts

## Step 2 - Check GPU (should print Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

# PART 1 - Detector cache (real images)
## Step 3 - Download DeepCrack
Expected counts: 300 / 300 / 237 / 237

In [ ]:
!git clone -q --depth 1 https://github.com/yhlleo/DeepCrack.git /content/DeepCrack
!unzip -q -o /content/DeepCrack/dataset/DeepCrack.zip -d /content/deepcrack_data
import os
for d in ['train_img','train_lab','test_img','test_lab']: print(d, len(os.listdir(f'/content/deepcrack_data/{d}')))

## Step 4 - Train U-Net + build cache (~10 min on T4)
If Colab disconnects after training finished, re-run with `--reuse_model`.

In [ ]:
!python scripts/build_detector_cache.py --data /content/deepcrack_data --out $DRIVE

## Step 5 - Check the detector (your slide material)
Uncertainty must go DOWN and IoU UP from q0 to q4. Then tell the team the cache is in Drive.

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
rep = json.load(open(f'{DRIVE}/detector_report.json'))
display(pd.DataFrame(rep['report']).round(3))
print('success threshold:', round(rep['success_unc'], 3), '| altitude->quality:', rep['alt_q'])
display(Image(f'{DRIVE}/detector_views.png'))
!python scripts/test_env.py | tail -1

# PART 2 - Agent C (A2C mission supervisor)
## Step 6 - Baselines (analytic)

In [ ]:
!python scripts/run_baselines.py --out $DRIVE/results

## Step 7 - Train Agent C (~3-5 min)

In [ ]:
!python scripts/train_agent_c.py --steps 100000 --out $DRIVE/agentC

## Step 8 - Learning curves + results table

In [ ]:
import glob
df = pd.concat([pd.read_csv(f, skiprows=1) for f in glob.glob(f'{DRIVE}/agentC/monitor/*.monitor.csv')]).sort_values('t')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(df['r'].rolling(100).mean().values); ax[0].set_title('Episode return (rolling 100)'); ax[0].set_xlabel('episode')
ax[1].plot(df['energy_used'].rolling(100).mean().values); ax[1].set_title('Energy used per mission')
ax[2].plot(df['defects_found'].rolling(100).mean().values); ax[2].set_title('Defects found per mission')
plt.tight_layout(); plt.savefig(f'{DRIVE}/agentC/learning_curves.png', dpi=150); plt.show()
pd.read_csv(f'{DRIVE}/agentC/agentC_vs_baselines.csv')[['policy','return_mean','success_mean','steps_mean','coverage_mean','defects_found_mean','miou_mean','energy_used_mean']].round(3)

## Step 9 - Live demo: state -> action probabilities -> option -> reward (CO3)

In [ ]:
import torch
from stable_baselines3 import A2C
from env import InspectionEnv
OPTS = ['Explore', 'Refine', 'Return home']
agent = A2C.load(f'{DRIVE}/agentC/best/best_model.zip', device='cpu')
env = InspectionEnv('supervise'); obs, info = env.reset(seed=1000)
for t in range(4):
    print(f'--- decision {t} | state [battery, coverage, mean_u, found, dist_home, time] = {np.round(obs, 2)}')
    with torch.no_grad():
        probs = agent.policy.get_distribution(agent.policy.obs_to_tensor(obs)[0]).distribution.probs[0].numpy()
    print('   policy pi(a|s):', {o: round(float(p), 2) for o, p in zip(OPTS, probs)})
    a = int(probs.argmax()); obs, r, term, trunc, info = env.step(a)
    print(f'   OPTION -> {OPTS[a]}   REWARD = {r:+.2f}   defects found = {info["defects_found"]}')
    if term or trunc: break

## Step 10 - Exploration sensitivity: entropy coefficient (CO5, ~10 min)

In [ ]:
for ent in [0.0, 0.01, 0.05]:
    !python scripts/train_agent_c.py --steps 60000 --ent_coef {ent} --out $DRIVE/agentC_ent{ent}

In [ ]:
rows = []
for ent in [0.0, 0.01, 0.05]:
    r = pd.read_csv(f'{DRIVE}/agentC_ent{ent}/agentC_vs_baselines.csv').iloc[0]
    rows.append({'ent_coef': ent, 'return': round(r.return_mean, 2), 'energy': round(r.energy_used_mean, 3),
                 'defects_found': r.defects_found_mean, 'decisions': r.steps_mean})
pd.DataFrame(rows)

## Step 11 (optional) - Agent C on real images, and with the trained Agent A inside 'explore'

In [ ]:
!python scripts/train_agent_c.py --detector learned --cache $DRIVE/detector_cache.npz --out $DRIVE/agentC_learned
!python scripts/train_agent_c.py --nav_model $DRIVE/agentA/best/best_model.zip --out $DRIVE/agentC_withA